In [7]:
import numpy as np
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from requests.adapters import HTTPAdapter
from requests.packages.urllib3.util.retry import Retry
import tqdm
import time
#set working directory
import os
import pycountry

# Define a proper function for requests with retry and custom headers
def requests_retry_session(retries=3, backoff_factor=0.3):
    session = requests.Session()
    # Set a proper user agent - THIS IS CRITICAL FOR NOMINATIM
    session.headers.update({
        'User-Agent': 'YourAppName/1.0 (your.email@example.com)',  # Use your actual info here
    })
    return session


# set working directory
os.chdir(r"C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\WFP")
def requests_retry_session(
    retries=3,
    backoff_factor=0.3,
    status_forcelist=(500, 502, 504),
    session=None,
):
    session = session or requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

def get_country_name(code):
    country = pycountry.countries.get(alpha_2=code)
    if country:
        return country.name
    else:
        return None
# read data
df = pd.read_csv(r"C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\WFP\Prices-Export-Tue Dec 02 2025 16_03_28 GMT-0500.csv")
# format Price Date to datetime
df['Price Date'] = pd.to_datetime(df['Price Date'])
# keep only Country, Admin 1, Admin 2, Market Name, Commodity, Price Date, Trend

df = df[['Country', 'Admin 1', 'Admin 2', 'Market Name', 'Commodity', 'Price Date', 'Trend']]
# rename Country to country_name, Admin 1 to admin1, Admin 2 to admin2, Market Name to market_name, Commodity to commodity_name, Price Date to price_date, Trend to trend
df = df.rename(columns={
    'Country': 'country_name',
    'Admin 1': 'admin1',
    'Admin 2': 'admin2',
    'Market Name': 'market_name',
    'Commodity': 'commodity_name',
    'Price Date': 'price_date',
    'Trend': 'trend'
})
# extract year and month from price_date
df['year'] = df['price_date'].dt.year
df['month'] = df['price_date'].dt.month
#drop price_date
df = df.drop(columns=['price_date'])
#drop commodity_name
df = df.drop(columns=['commodity_name'])
# group by country_name, admin1, admin2, market_name, year, month, aggregate the mean and std of trend
df = df.groupby(['country_name','year', 'month']).agg(
    trend_mean=('trend', 'mean'),
    trend_std=('trend', 'std')
).reset_index()
# create an address column combining country_name, admin1, admin2, market_name (using , as separator)
#drop NAs in trend
df = df.dropna(subset=['trend_mean'])
#rename trend to WFP_Price
df = df.rename(columns={'trend_mean': 'WFP_Price', 'trend_std': 'WFP_Price_std'})


C:\Users\swl00\AppData\Local\Temp\ipykernel_18864\3591049453.py:55: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Price Date'] = pd.to_datetime(df['Price Date'])


In [8]:
# convert country_name to three digit code
def convert_country_name_to_code(country_name):
    try:
        # Get the country object using the name
        country = pycountry.countries.lookup(country_name)
        # Return the alpha_3 code
        return country.alpha_3
    except LookupError:
        # If the country name is not found, return None or handle it as needed
        return None
    
# Apply the function to the 'country_name' column
df['country_code'] = df['country_name'].apply(convert_country_name_to_code)

In [9]:
# rename country_code to country_code_3
df = df.rename(columns={'country_code': 'country_code_3'})

In [10]:
#read IPC_scaffold
df_scaffold = pd.read_csv("ch_scaffold.csv")

In [11]:
# drop duplicates on lat, lon, year and month
df_scaffold = df_scaffold.drop_duplicates(subset=['lat', 'lon', 'year', 'month'])

In [12]:
# lat ad lon keep 12 digits, padding with 0
df_scaffold['lat'] = df_scaffold['lat'].apply(lambda x: '{:.12f}'.format(x))
df_scaffold['lon'] = df_scaffold['lon'].apply(lambda x: '{:.12f}'.format(x))

In [13]:
# read df_ESA
df_ESA = pd.read_csv(r"C:\Users\swl00\IFPRI Dropbox\Weilun Shi\Google fund\Analysis\1.Source Data\ESA\ch_scaffold_fixed.csv")

In [14]:
df_ESA['country'].unique()

array(['Mali', 'Niger', 'Burkina Faso', 'Chad', 'Senegal', 'Nigeria',
       'Benin', 'Guinea', 'Sierra Leone', 'Cameroon', 'Mauritania',
       'Ghana', "Cote d'Ivoire", 'Guinea-Bissau',
       'Central African Republic', 'Togo', 'Cabo Verde', 'Liberia',
       'Gambia'], dtype=object)

In [15]:
# convert lat and lon to float and then padding to 12 digits
df_ESA['lat'] = df_ESA['lat'].astype(float).apply(lambda x: '{:.12f}'.format(x))
df_ESA['lon'] = df_ESA['lon'].astype(float).apply(lambda x: '{:.12f}'.format(x))

In [16]:
# for df_ESA, only keep ISO3, country, lat and lon
df_ESA = df_ESA[['ISO3', 'country', 'title', 'lat', 'lon']]

# drop duplicates
df_ESA = df_ESA.drop_duplicates(subset=['lat', 'lon'])

In [17]:
# merge on lat and lon
df_scaffold = df_scaffold.merge(df_ESA, on=['title'], how='left', suffixes=('', '_fixed'))

In [18]:
import pycountry
df_scaffold['country_code_3'] = df_scaffold['ISO3']

In [19]:
# convert year and month to integer
df_scaffold['year'] = df_scaffold['year'].astype(int)
df_scaffold['month'] = df_scaffold['month'].astype(int)

In [20]:
# merge df_scaffold and df on country_code_3, year and month
df_scaffold = df_scaffold.merge(df, on=['country_code_3', 'year', 'month'], how='left', indicator=True)

In [21]:
# see _merge
df_scaffold['_merge'].value_counts()

_merge
both          186938
left_only      61282
right_only         0
Name: count, dtype: int64

In [22]:
# drop _merge
df_scaffold = df_scaffold.drop(columns=['_merge'])

In [23]:
# drop country_code_3 and country_name
df_scaffold = df_scaffold.drop(columns=['country_code_3'])

# drop any duplicates in lat lon, year and month
df_scaffold = df_scaffold.drop_duplicates(subset=['lat', 'lon', 'year', 'month'])

In [24]:
df_scaffold.describe()

,year,month,area_id,WFP_Price,WFP_Price_std
count,238500.000000,238500.00000,238500.000000,177960.000000,167099.000000
mean,2017.000000,6.50000,662.000000,973.470479,406.522859
std,4.320503,3.45206,382.495246,1727.488800,1007.504643
min,2010.000000,1.00000,0.000000,2.583378,0.360978
25%,2013.000000,3.75000,331.000000,353.304658,82.648118
50%,2017.000000,6.50000,662.000000,538.091490,180.042357
75%,2021.000000,9.25000,993.000000,897.221760,397.839100
max,2024.000000,12.00000,1324.000000,24494.916596,14191.127844


In [25]:
# location is the unique lat and lon
locations = df_scaffold[['lat', 'lon']].drop_duplicates()

In [26]:
len(locations)

1325

In [27]:
# export
df_scaffold.to_csv(r"ch_WFP_prices.csv", index=False)